In [ ]:
# Import relevant packages
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import ruptures as rpt

import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
# --- Temporal Trends Analysis with ITS ---

# --- DEFINE VARIABLES ---
fname = 'imputed_patient_profile_analysis.csv'
df_patient_profiles = pd.read_csv(fname)

print(df_patient_profiles.columns)
print(len(df_patient_profiles))

# Load relevant columns from your dataset
df_patient_profiles = df_patient_profiles[['patientdurablekey','startdatekey_firstLBP', 'startdatekey', 'opioid']].copy()
df_patient_profiles = df_patient_profiles[df_patient_profiles['startdatekey_firstLBP']>20120000]

df_patient_profiles.head()

# Change point analysis

## Quarterly

### Prep

In [ ]:
# Extract year and quarter from date
df = pd.copy(df_patient_profiles, deep=True)
df['date_firstLBP'] = pd.to_datetime(df['startdatekey_firstLBP'], format='%Y%m%d')
df['year_quarter_firstLBP'] = df['date_firstLBP'].dt.to_period('Q')  # Quarterly periods

# Plot histogram of LBP records by quarter
plt.figure(figsize=(15, 6))
df['year_quarter_firstLBP'].value_counts().sort_index().plot(kind='bar')
plt.title('Count of LBP Records by Quarter')
plt.xlabel('Year-Quarter')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Process opioid records
df2 = df[df['startdatekey'] > 0]
df2['date_opioid'] = pd.to_datetime(df2['startdatekey'], format='%Y%m%d')
df2['year_quarter_opioid'] = df2['date_opioid'].dt.to_period('Q')

# Plot histogram of opioid records by quarter
plt.figure(figsize=(15, 6))
df2['year_quarter_opioid'].value_counts().sort_index().plot(kind='bar')
plt.title('Count of Opioid Records by Quarter')
plt.xlabel('Year-Quarter')
plt.ylabel('Count')
plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate opioid prescription rate per quarter
df['year_quarter'] = pd.to_datetime(df['startdatekey_firstLBP'], format='%Y%m%d').dt.to_period('Q')
opioid_counts_quarterly = df[df['opioid'] == True].groupby(df['year_quarter']).size()
total_counts_quarterly = df.groupby(df['year_quarter']).size()
prescription_rate_quarterly = (opioid_counts_quarterly / total_counts_quarterly).fillna(0)

# Convert to time series for ruptures
ts_quarterly = prescription_rate_quarterly.reset_index()
ts_quarterly.columns = ['year_quarter', 'rate']
# Convert period to datetime for plotting (use the start of each quarter)
ts_quarterly['date'] = ts_quarterly['year_quarter'].dt.to_timestamp()

# Plot the quarterly rates
plt.figure(figsize=(15, 6))
plt.plot(ts_quarterly['date'], ts_quarterly['rate'], marker='o')
plt.axvline(pd.to_datetime('2016-01-01'), color='red', linestyle='--')
plt.axvline(pd.to_datetime('2022-01-01'), color='red', linestyle='--')
plt.title('Opioid Prescription Rate by Quarter (Raw Data)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

### Change Point Algorithm

In [ ]:
signal_quarterly = ts_quarterly['rate'].values

# Dynamic programing
algo = rpt.Dynp(model='rbf', min_size=2, jump=1).fit(signal_quarterly)  
change_points = algo.predict(n_bkps=2)
print(change_points)

# Get change point quarters
change_quarters = ts_quarterly.iloc[change_points[:-1]]['year_quarter']
print("Detected change quarters:", change_quarters.values)

# Visualize the change points
plt.figure(figsize=(15, 6))
plt.plot(ts_quarterly['date'], ts_quarterly['rate'], marker='o')

# Mark the detected change points (using the dynamic programming results)
for cp in change_points[:-1]:
    plt.axvline(ts_quarterly['date'].iloc[cp], color='green', linestyle='--', 
                label=f'Change point: {ts_quarterly["year_quarter"].iloc[cp]}')

plt.title('Opioid Prescription Rate by Quarter with Detected Change Points')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Use PELT algorithm (adjust penalty `pen` for sensitivity)
penalties = np.logspace(-1, 2, 20)  # Test penalties from 0.1 to 100 (log scale)
n_changes = []

n_points = -1
for pen in penalties:
    # PELT (auto-detects number of changes)
    algo = rpt.Pelt(model="rbf", min_size=8).fit(signal_quarterly)  
    change_points = algo.predict(pen=pen)
    if len(change_points) != n_points:
        # Get change point quarters
        change_quarters = ts_quarterly.iloc[change_points[:-1]]['year_quarter']
        print(pen, len(change_points), "Detected change quarters:", change_quarters.values)
        n_points = len(change_points)
    n_changes.append(len(change_points) - 1)  # Subtract 1 (last point is end of series)

# Plot results
plt.figure(figsize=(10, 4))
#plt.semilogx(penalties, n_changes, marker="o")
plt.plot(penalties, n_changes, marker="o")
plt.xlabel("Penalty (log scale)")
plt.ylabel("Number of Change Points")
plt.title("Elbow Method for Penalty Selection")
plt.grid(True)
plt.show()


In [ ]:
# PELT (auto-detects number of changes)
algo = rpt.Pelt(model="rbf", min_size=8).fit(signal_quarterly)  
change_points = algo.predict(pen=2)
if len(change_points) != n_points:
    # Get change point quarters
    change_quarters = ts_quarterly.iloc[change_points[:-1]]['year_quarter']

# Visualize the change points
plt.figure(figsize=(15, 6))
plt.plot(ts_quarterly['date'], ts_quarterly['rate'], marker='o')

# Mark the detected change points (using the dynamic programming results)
for cp in change_points[:-1]:
    plt.axvline(ts_quarterly['date'].iloc[cp], color='green', linestyle='--', 
                label=f'Change point: {ts_quarterly["year_quarter"].iloc[cp]}')

plt.title('Opioid Prescription Rate by Quarter with Detected Change Points')
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Annually

### Prep

In [ ]:
df = pd.copy(df_patient_profiles, deep=True)
# Extract year (first 4 digits)
start_year = df['startdatekey_firstLBP'].astype(str).str[:4].astype(int)

# Plot histogram
#start_year.value_counts().sort_index().plot(kind='bar', figsize=(10, 6), rot=0)
plt.figure(figsize=(10, 6))
plt.hist(start_year, bins=range(2010, 2024 + 1), edgecolor='black', align='left')

plt.title('Count of LBP Records by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

# Extract year (first 4 digits)
df2 = df[df['startdatekey']>0]
start_year = df2['startdatekey'].astype(str).str[:4].astype(int)

# Plot histogram
#start_year.value_counts().sort_index().plot(kind='bar', figsize=(10, 6), rot=0)
plt.figure(figsize=(10, 6))
plt.hist(start_year, bins=range(2010, 2024 + 1), edgecolor='black', align='left')

plt.title('Count of opioid Records by Year')
plt.xlabel('Year')
plt.ylabel('Count')
plt.show()

In [ ]:
# Select the relevant columns

# Convert year to datetime and count opioid prescriptions per year
df["year"] = pd.to_datetime(df["startdatekey_firstLBP"], format='%Y%m%d')  # Convert to datetime
opioid_counts = df[df["opioid"] == True].groupby(df["year"].dt.year).size()
total_counts = df.groupby(df["year"].dt.year).size()
prescription_rate = (opioid_counts / total_counts).fillna(0)  # Rate per year

# Convert to time series for ruptures
ts = prescription_rate.reset_index()
ts.columns = ["year", "rate"]
ts["year"] = pd.to_datetime(ts["year"], format="%Y")  # Year as datetime

plt.plot(ts["year"], ts["rate"], marker="o")
plt.axvline(pd.to_datetime("2016"), color="red", linestyle="--")
plt.axvline(pd.to_datetime("2022"), color="red", linestyle="--")
plt.title("Opioid Prescription Rate (Raw Data)")
plt.show()

### Change Point Algorithm

In [ ]:
signal = ts["rate"].values

# PELT (auto-detects number of changes)
algo = rpt.Pelt(model="rbf", min_size=2).fit(signal)  
change_points = algo.predict(pen=0.1)  # No need to predefine count!
print(change_points)

# Get change point years
change_years = ts.iloc[change_points[:-1]]["year"].dt.year
print("Detected change years:", change_years.values)

# Use PELT algorithm (adjust penalty `pen` for sensitivity)
algo = rpt.Pelt(model="rbf").fit(signal)
change_points = algo.predict(pen=1)  # Lower pen = more sensitive
print(change_points)

# Get change point years
change_years = ts.iloc[change_points[:-1]]["year"].dt.year
print("Detected change years:", change_years.values)

# Dynamic programing
algo = rpt.Dynp(model="rbf", min_size=2, jump=1).fit(signal)  
change_points = algo.predict(n_bkps=2)  # Must specify "2 changes"
print(change_points)

# Get change point years
change_years = ts.iloc[change_points[:-1]]["year"].dt.year
print("Detected change years:", change_years.values)

In [ ]:
penalties = np.logspace(-1, 2, 20)  # Test penalties from 0.1 to 100 (log scale)
n_changes = []

for pen in penalties:
    # PELT (auto-detects number of changes)
    algo = rpt.Pelt(model="rbf", min_size=2).fit(signal)  
    change_points = algo.predict(pen=pen)
    n_changes.append(len(change_points) - 1)  # Subtract 1 (last point is end of series)

# Plot results
plt.figure(figsize=(10, 4))
plt.semilogx(penalties, n_changes, marker="o")
plt.xlabel("Penalty (log scale)")
plt.ylabel("Number of Change Points")
plt.title("Elbow Method for Penalty Selection")
plt.grid(True)
plt.show()


# Interrupted Time Series Analysis (ITS)

In [ ]:
# --- DATA PREP ---
# Load relevant columns from your dataset
df = df_patient_profiles[['patientdurablekey','startdatekey_firstLBP', 'opioid']].copy()
df = df[df['startdatekey_firstLBP']>20120000]

# Ensure LBP_startdatekey is in datetime format
df['startdatekey_firstLBP'] = df['startdatekey_firstLBP'].astype(str)
df['LBP_date'] = pd.to_datetime(df['startdatekey_firstLBP'], format='%Y%m%d', errors='coerce')

# Assign each row to a quarter (start of quarter)
df['quarter'] = df['LBP_date'].dt.to_period('Q').dt.start_time

# Sort and create a time index
df = df.sort_values("quarter").reset_index(drop=True)
df['quarter_num'] = df['quarter'].rank(method='dense').astype(int) - 1  # 0-indexed

# Define policy cutoffs (change points)
#DYNAP
cutoff1 = pd.to_datetime("2015-04-01") # updated 3/15/2025
cutoff2 = pd.to_datetime("2018-10-01") # updated 11/4/2022
# # PELT
# cutoff1 = pd.to_datetime("2014-07-01") # updated 3/15/2025
# cutoff2 = pd.to_datetime("2018-04-01") # updated 11/4/2022

# Segment indicators
df['segment_2015Q2'] = (df['quarter'] >= cutoff1).astype(int)
df['segment_2018Q4'] = (df['quarter'] >= cutoff2).astype(int)

# Time since each intervention (slope change terms)
df['quarters_since_2015Q2'] = (df['quarter_num'] - df.loc[df['quarter'] >= cutoff1, 'quarter_num'].min()).clip(lower=0)
df['quarters_since_2018Q4'] = (df['quarter_num'] - df.loc[df['quarter'] >= cutoff2, 'quarter_num'].min()).clip(lower=0)

# Seasonality: quarter of year (0 to 3)
df['quarter_of_year'] = df['LBP_date'].dt.quarter - 1
df['sin4'] = np.sin(2 * np.pi * df['quarter_of_year'] / 4)
df['cos4'] = np.cos(2 * np.pi * df['quarter_of_year'] / 4)

df


In [ ]:
# --- Fit logistic model ---
# Define X and y
X_with_season = sm.add_constant(df[[
    'quarter_num',           # baseline slope
    'segment_2015Q2',          # level change 2017
    'quarters_since_2015Q2',   # slope change post-2017
    'segment_2018Q4',          # level change 2023
    'quarters_since_2018Q4',   # slope change post-2023
    'sin4', 'cos4'           # seasonality
]])
y = df['opioid']  # binary outcome

# Fit logistic regression model
logit_model = sm.GLM(y, X_with_season, family=sm.families.Binomial())
logit_result = logit_model.fit()

# Output summary
print("\nModel WITH Seasonality:")
print(logit_result.summary())

# Predict with confidence intervals
pred = logit_result.get_prediction(X_with_season)
pred_summary = pred.summary_frame(alpha=0.05)

df['predicted_prob'] = pred_summary['mean']
df['ci_lower'] = pred_summary['mean_ci_lower']
df['ci_upper'] = pred_summary['mean_ci_upper']


# --- Predict without accounting for the season ---
X_without_season = sm.add_constant(df[[
    'quarter_num',           # baseline slope
    'segment_2015Q2',          # level change 2017
    'quarters_since_2015Q2',   # slope change post-2017
    'segment_2018Q4',          # level change 2023
    'quarters_since_2018Q4',   # slope change post-2023
]])

# Fit logistic regression model
logit_model_without_season = sm.GLM(y, X_without_season, family=sm.families.Binomial())
logit_result_without_season = logit_model_without_season.fit()

print("\nModel WITHOUT Seasonality:")
print(logit_result_without_season.summary())

# Predict with confidence intervals
pred_without_season = logit_result_without_season.get_prediction(X_without_season)
pred_summary_without_season = pred_without_season.summary_frame(alpha=0.05)

df['predicted_prob_without_season'] = pred_summary_without_season['mean']
df['ci_lower_without_season'] = pred_summary_without_season['mean_ci_lower']
df['ci_upper_without_season'] = pred_summary_without_season['mean_ci_upper']

# Model comparison
print("\nAIC Comparison:")
print(f"AIC with seasonality:    {logit_result.aic:.2f}")
print(f"AIC without seasonality: {logit_result_without_season.aic:.2f}")





In [ ]:
# --- Aggregate and plot ---
summary_df = df.groupby('quarter').agg(
    actual_rate=('opioid', 'mean'),
    predicted_rate=('predicted_prob', 'mean'),
    ci_lower=('ci_lower', 'mean'),
    ci_upper=('ci_upper', 'mean'),
    count=('opioid', 'count'),
    predicted_rate_without_season=('predicted_prob_without_season', 'mean'),
    ci_lower_without_season=('ci_lower_without_season', 'mean'),
    ci_upper_without_season=('ci_upper_without_season', 'mean'),
).reset_index()

plt.figure(figsize=(12, 6))
plt.plot(summary_df['quarter'], summary_df['actual_rate'], label='Actual Opioid Rate', marker='o', color='green')
# Add intervention lines
plt.axvline(cutoff1, color='red', linestyle=':', label='2015Q2 Intervention')
plt.axvline(cutoff2, color='red', linestyle=':', label='2018Q4 Intervention')
# With Seasons (fit line and confidence interval)
plt.plot(summary_df['quarter'], summary_df['predicted_rate'], label='Predicted Rate (Seasons)', linestyle='--', color='blue')
plt.fill_between(summary_df['quarter'], summary_df['ci_lower'], summary_df['ci_upper'], color='blue', alpha=0.2, label='95% CI (Seasons)')
# Without Seasons (fit line and confidence interval)
plt.plot(summary_df['quarter'], summary_df['predicted_rate_without_season'], label='Predicted (No Seasons)', linestyle='-.', color='orange')
plt.fill_between(summary_df['quarter'], summary_df['ci_lower_without_season'], summary_df['ci_upper_without_season'], color='orange', alpha=0.2, label='95% CI (No Seasons)')

plt.title('Opioid Prescription Rate Over Time')
plt.ylabel('Rate')
plt.xlabel('Quarter')
plt.legend()
plt.grid(True)
plt.minorticks_on()
plt.tight_layout()
plt.savefig('any_annual_opioid_prescription_rate.tiff', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Get residuals from the fitted model
residuals = logit_result.resid_response

# --- Durbin-Watson statistic ---
dw_stat = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw_stat:.2f}')

# --- ACF and PACF plots ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(residuals, lags=24, ax=ax[0])
plot_pacf(residuals, lags=24, ax=ax[1])
ax[0].set_title('ACF of Residuals')
ax[1].set_title('PACF of Residuals')
plt.tight_layout()
plt.show()


# Opioid Prescription Strength (morphine miligram equivalents - MMEs)

## Prepare data

In [ ]:
import os
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

import pyrootutils
root = pyrootutils.setup_root(
    search_from=os.path.abspath(''),
    indicator=[".git"],
    pythonpath=True, # add root directory to the PYTHONPATH (helps with imports)
    dotenv=True, # load environment variables from .env if exists in root directory
)
from utils.extractor import Extractor

PlumsExtractor = Extractor(num_results_flag = True, display_results_flag = True)


In [ ]:
# --- Temporal Trends Analysis with Opioid Dosage (MME) ---

# --- DEFINE VARIABLES ---

fname = 'imputed_patient_profile_analysis.csv'
df_patient_profiles = pd.read_csv(fname)
print(df_patient_profiles.columns)
print(len(df_patient_profiles))

# Load relevant columns from your dataset
df_patient_profiles = df_patient_profiles[['patientdurablekey','startdatekey_firstLBP', 'startdatekey', 'opioid']].copy()
df_patient_profiles = df_patient_profiles[df_patient_profiles['startdatekey_firstLBP']>20120000]

#Identify patients who recieved an opioid
df_patients_with_opioid = df_patient_profiles[df_patient_profiles['opioid']==1].reset_index(drop=True)
patientdurablekey_list = df_patients_with_opioid['patientdurablekey'].drop_duplicates().tolist()

print(len(df_patients_with_opioid))
print(len(patientdurablekey_list))

df_patients_with_opioid.head()

In [ ]:
# --- DEFINE VARIABLES ---
med_info_df_fname = 'medicationorderfact.csv'
df_med_info = pd.read_csv(med_info_df_fname)

print(df_med_info.columns)
print(len(df_med_info))

df_med_info.head()

In [ ]:
# MED CONTEXT NEEDED FOR MME CONVERSION
# TODO (Manual review required): Save information to a csv and extract the relevant dosage, frequency, and conversion information

medQuery = f'''
/*
Created By: Michelle Tong
Description: Select opioid med details for MME conversions
*/

SELECT DISTINCT 
  a.medicationname,
  -- a.medicationform,
  a.medicationstrength,
  a.medicationroute,
  -- a.frequency
FROM df_med_info AS a
RIGHT JOIN 
    df_patients_with_opioid AS b
ON 
    a.patientdurablekey = b.patientdurablekey
WHERE 
  a.patientdurablekey IN {tuple(patientdurablekey_list)}
  AND a.startdatekey BETWEEN b.startdatekey AND b.startdatekey + 1
ORDER BY a.medicationname
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(medQuery, runtime_flag=True)

# results_df_pd.to_csv('medication_names_for_MME_dosage.csv')

# --- DEFINE VARIABLES ---
med_mme_conversions_df_fname = 'conversions_MME_for_ITS.xlsx'
df_med_conversions_mg = pd.read_excel(med_mme_conversions_df_fname, sheet_name='mg_info')
df_med_conversions_freq = pd.read_excel(med_mme_conversions_df_fname, sheet_name='frequency_info')
df_med_conversions_morphine = pd.read_excel(med_mme_conversions_df_fname, sheet_name='morphine_equivalent')

print(df_med_conversions_mg.columns)
print(len(df_med_conversions_mg))

df_med_conversions_mg.head()

results_df_pd

In [ ]:
# MED INFO

medQuery = f'''
/*
Created By: Michelle Tong
Description: Select opioid med details for patients with LBP
*/

SELECT DISTINCT 
  a.patientdurablekey, 
  a.medicationname, 
  -- a.medicationtherapeuticclass, -- all analgesics
  -- a.medicationpharmaceuticalclass, -- all contain the words "opioid" and "analgesic"
  a.medicationstrength, 
  -- a.medicationform, level of specificity is not needed, instead using medicationroute
  a.medicationroute,
  -- a.route, -- medicationroute has more field entries
  a.frequency,
  a.doseunit,
  a.startdatekey,
  b.startdatekey,
  -- a.class,
  a.opioidtype,
  c.MG,
  d.times_per_day,
  e.Conversion_ratio_for_MME_NIH_HEAL_2024
FROM df_med_info AS a
RIGHT JOIN 
    df_patients_with_opioid AS b
  ON a.patientdurablekey = b.patientdurablekey
LEFT JOIN
    df_med_conversions_mg AS c
  ON a.medicationname = c.medicationname
LEFT JOIN
    df_med_conversions_freq AS d
  ON a.frequency = d.frequency
LEFT JOIN
    df_med_conversions_morphine AS e
  ON a.medicationname LIKE '%' || e.Opioid_Agonist || '%'
WHERE 
  a.patientdurablekey IN {tuple(patientdurablekey_list)}
  AND a.startdatekey BETWEEN b.startdatekey AND b.startdatekey + 0
  AND LOWER(a.frequency) NOT LIKE '%unspecified%'
ORDER BY a.patientdurablekey, a.startdatekey
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(medQuery, runtime_flag=True)
results_df_pd['MME'] = results_df_pd['MG'] * results_df_pd['times_per_day'] * results_df_pd['Conversion_ratio_for_MME_NIH_HEAL_2024']
df_med_dosage = results_df_pd.copy()
print(df_med_dosage['opioidtype'].value_counts())

# Aggregate MME by patientdurablekey
df_med_aggregated = df_med_dosage.groupby('patientdurablekey', as_index=False).agg({
    'MME': 'sum',
    'startdatekey': 'min'   # earliest date
})

df_med_aggregated['MME_above_50'] = df_med_aggregated['MME'] > 50
print(len(df_med_aggregated))
print(len(df_med_aggregated[df_med_aggregated['MME']>50]))


In [ ]:
# MED HIGH DOSAGE DEMOGRAPHICS
fname = 'imputed_patient_profile_analysis.csv'
df_patient_profiles = pd.read_csv(fname)

medQuery = f'''
/*
Created By: Michelle Tong
Description: Select patient demographics
*/

SELECT DISTINCT 
  a.patientdurablekey, 
  a.age_firstLBP,
  a.sex,
  a.obesity,
  a.raceethnicity,
  b.MME_above_50, 
 
FROM df_patient_profiles AS a
RIGHT JOIN 
    df_med_aggregated AS b
  ON a.patientdurablekey = b.patientdurablekey
WHERE 
  a.patientdurablekey IN {tuple(patientdurablekey_list)}
  AND b.MME_above_50 = True
ORDER BY a.patientdurablekey
'''

# Run query and update relevant keys
results_df_pd = PlumsExtractor.run_query(medQuery, runtime_flag=True)

print(np.mean(results_df_pd['age_firstLBP']))
print(np.std(results_df_pd['age_firstLBP']))
print(results_df_pd['sex'].value_counts())
print(results_df_pd['obesity'].value_counts())
print(results_df_pd['raceethnicity'].value_counts())


## ITS with Medication Dosage in MMEs

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.stattools import durbin_watson
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

### Quarterly

In [ ]:
# --- DATA PREP ---
# Load relevant columns from your dataset
df = df_med_aggregated[['patientdurablekey','startdatekey', 'MME_above_50']].copy()

# Ensure LBP_startdatekey is in datetime format
df['startdatekey'] = df['startdatekey'].astype(str)
df['med_date'] = pd.to_datetime(df['startdatekey'], format='%Y%m%d', errors='coerce')

# Assign each row to a quarter (start of quarter)
df['quarter'] = df['med_date'].dt.to_period('Q').dt.start_time

# Sort and create a time index
df = df.sort_values("quarter").reset_index(drop=True)
df['quarter_num'] = df['quarter'].rank(method='dense').astype(int) - 1  # 0-indexed

# Define policy cutoffs (change points)
#DYNAP
cutoff1 = pd.to_datetime("2015-04-01") # updated 3/15/2025
cutoff2 = pd.to_datetime("2018-10-01") # updated 11/4/2022

# Segment indicators
df['segment_2015Q2'] = (df['quarter'] >= cutoff1).astype(int)
df['segment_2018Q4'] = (df['quarter'] >= cutoff2).astype(int)

# Time since each intervention (slope change terms)
df['quarters_since_2015Q2'] = (df['quarter_num'] - df.loc[df['quarter'] >= cutoff1, 'quarter_num'].min()).clip(lower=0)
df['quarters_since_2018Q4'] = (df['quarter_num'] - df.loc[df['quarter'] >= cutoff2, 'quarter_num'].min()).clip(lower=0)

# Seasonality: quarter of year (0 to 3)
df['quarter_of_year'] = df['med_date'].dt.quarter - 1
df['sin4'] = np.sin(2 * np.pi * df['quarter_of_year'] / 4)
df['cos4'] = np.cos(2 * np.pi * df['quarter_of_year'] / 4)

df


In [ ]:
# --- Fit logistic model ---
# Define X and y
X_with_season = sm.add_constant(df[[
    'quarter_num',           # baseline slope
    'segment_2015Q2',          # level change 2017
    'quarters_since_2015Q2',   # slope change post-2017
    'segment_2018Q4',          # level change 2023
    'quarters_since_2018Q4',   # slope change post-2023
    'sin4', 'cos4'           # seasonality
]])
y = df['MME_above_50']  # binary outcome

# Fit logistic regression model
logit_model = sm.GLM(y, X_with_season, family=sm.families.Binomial())
logit_result = logit_model.fit()

# Output summary
print("\nModel WITH Seasonality:")
print(logit_result.summary())

# Predict with confidence intervals
pred = logit_result.get_prediction(X_with_season)
pred_summary = pred.summary_frame(alpha=0.05)

df['predicted_prob'] = pred_summary['mean']
df['ci_lower'] = pred_summary['mean_ci_lower']
df['ci_upper'] = pred_summary['mean_ci_upper']


# --- Predict without accounting for the season ---
X_without_season = sm.add_constant(df[[
    'quarter_num',           # baseline slope
    'segment_2015Q2',          # level change 2017
    'quarters_since_2015Q2',   # slope change post-2017
    'segment_2018Q4',          # level change 2023
    'quarters_since_2018Q4',   # slope change post-2023
]])

# Fit logistic regression model
logit_model_without_season = sm.GLM(y, X_without_season, family=sm.families.Binomial())
logit_result_without_season = logit_model_without_season.fit()

print("\nModel WITHOUT Seasonality:")
print(logit_result_without_season.summary())

# Predict with confidence intervals
pred_without_season = logit_result_without_season.get_prediction(X_without_season)
pred_summary_without_season = pred_without_season.summary_frame(alpha=0.05)

df['predicted_prob_without_season'] = pred_summary_without_season['mean']
df['ci_lower_without_season'] = pred_summary_without_season['mean_ci_lower']
df['ci_upper_without_season'] = pred_summary_without_season['mean_ci_upper']

# Model comparison
print("\nAIC Comparison:")
print(f"AIC with seasonality:    {logit_result.aic:.2f}")
print(f"AIC without seasonality: {logit_result_without_season.aic:.2f}")



In [ ]:
# --- Aggregate and plot ---
summary_df = df.groupby('quarter').agg(
    actual_rate=('MME_above_50', 'mean'),
    predicted_rate=('predicted_prob', 'mean'),
    ci_lower=('ci_lower', 'mean'),
    ci_upper=('ci_upper', 'mean'),
    count=('MME_above_50', 'count'),
    predicted_rate_without_season=('predicted_prob_without_season', 'mean'),
    ci_lower_without_season=('ci_lower_without_season', 'mean'),
    ci_upper_without_season=('ci_upper_without_season', 'mean'),
).reset_index()

plt.figure(figsize=(12, 6))
plt.plot(summary_df['quarter'], summary_df['actual_rate'], label='Actual Opioid Rate', marker='o', color='green')
# Add intervention lines
plt.axvline(cutoff1, color='red', linestyle=':', label='2015Q2 Intervention')
plt.axvline(cutoff2, color='red', linestyle=':', label='2018Q4 Intervention')
# # With Seasons (fit line and confidence interval)
# plt.plot(summary_df['quarter'], summary_df['predicted_rate'], label='Predicted Rate (Seasons)', linestyle='--', color='blue')
# plt.fill_between(summary_df['quarter'], summary_df['ci_lower'], summary_df['ci_upper'], color='blue', alpha=0.2, label='95% CI (Seasons)')
# Without Seasons (fit line and confidence interval)
plt.plot(summary_df['quarter'], summary_df['predicted_rate_without_season'], label='Predicted (No Seasons)', linestyle='-.', color='orange')
plt.fill_between(summary_df['quarter'], summary_df['ci_lower_without_season'], summary_df['ci_upper_without_season'], color='orange', alpha=0.2, label='95% CI (No Seasons)')

plt.title('Opioid Prescription Rate Over Time')
plt.ylabel('Rate')
plt.xlabel('Quarter')
plt.legend()
plt.grid(True)
plt.minorticks_on()
plt.tight_layout()
plt.show()

In [ ]:
# Get residuals from the fitted model
residuals = logit_result.resid_response

# --- Durbin-Watson statistic ---
dw_stat = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw_stat:.2f}')

# --- ACF and PACF plots ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(residuals, lags=24, ax=ax[0])
plot_pacf(residuals, lags=24, ax=ax[1])
ax[0].set_title('ACF of Residuals')
ax[1].set_title('PACF of Residuals')
plt.tight_layout()
plt.show()


In [ ]:
# Coeff of Variation
np.std(summary_df['actual_rate'])/np.mean(summary_df['actual_rate'])

### Annually

In [ ]:
# --- DATA PREP ---
# Load relevant columns from your dataset
df = df_med_aggregated[['patientdurablekey','startdatekey', 'MME_above_50']].copy()

# Ensure LBP_startdatekey is in datetime format
df['startdatekey'] = df['startdatekey'].astype(str)
df['med_date'] = pd.to_datetime(df['startdatekey'], format='%Y%m%d', errors='coerce')

# Assign each row to a year (start of year)
df['year'] = df['med_date'].dt.to_period('Y').dt.start_time

# Sort and create a time index
df = df.sort_values("year").reset_index(drop=True)
df['year_num'] = df['year'].rank(method='dense').astype(int) - 1  # 0-indexed

# Define policy cutoffs (change points)
#DYNAP
cutoff1 = pd.to_datetime("2015") # updated 11/5/2025
cutoff2 = pd.to_datetime("2019") # updated 11/5/2025

# Segment indicators
df['segment_2015'] = (df['year'] >= cutoff1).astype(int)
df['segment_2019'] = (df['year'] >= cutoff2).astype(int)

# Time since each intervention (slope change terms)
df['years_since_2015'] = (df['year_num'] - df.loc[df['year'] >= cutoff1, 'year_num'].min()).clip(lower=0)
df['years_since_2019'] = (df['year_num'] - df.loc[df['year'] >= cutoff2, 'year_num'].min()).clip(lower=0)

# Seasonality: quarter of year (0 to 3)
df['year_of_years'] = df['med_date'].dt.quarter - 1
df['sin4'] = np.sin(2 * np.pi * df['year_of_years'] / 4)
df['cos4'] = np.cos(2 * np.pi * df['year_of_years'] / 4)

df


In [ ]:
# --- Fit logistic model ---
# Define X and y
X_with_season = sm.add_constant(df[[
    'year_num',           # baseline slope
    'segment_2015',          # level change 2017
    'years_since_2015',   # slope change post-2017
    'segment_2019',          # level change 2023
    'years_since_2019',   # slope change post-2023
    'sin4', 'cos4'           # seasonality
]])
y = df['MME_above_50']  # binary outcome

# Fit logistic regression model
logit_model = sm.GLM(y, X_with_season, family=sm.families.Binomial())
logit_result = logit_model.fit()

# Output summary
print("\nModel WITH Seasonality:")
print(logit_result.summary())

# Predict with confidence intervals
pred = logit_result.get_prediction(X_with_season)
pred_summary = pred.summary_frame(alpha=0.05)

df['predicted_prob'] = pred_summary['mean']
df['ci_lower'] = pred_summary['mean_ci_lower']
df['ci_upper'] = pred_summary['mean_ci_upper']


# --- Predict without accounting for the season ---
X_without_season = sm.add_constant(df[[
    'year_num',           # baseline slope
    'segment_2015',          # level change 2017
    'years_since_2015',   # slope change post-2017
    'segment_2019',          # level change 2023
    'years_since_2019',   # slope change post-2023
]])

# Fit logistic regression model
logit_model_without_season = sm.GLM(y, X_without_season, family=sm.families.Binomial())
logit_result_without_season = logit_model_without_season.fit()

print("\nModel WITHOUT Seasonality:")
print(logit_result_without_season.summary())

# Predict with confidence intervals
pred_without_season = logit_result_without_season.get_prediction(X_without_season)
pred_summary_without_season = pred_without_season.summary_frame(alpha=0.05)

df['predicted_prob_without_season'] = pred_summary_without_season['mean']
df['ci_lower_without_season'] = pred_summary_without_season['mean_ci_lower']
df['ci_upper_without_season'] = pred_summary_without_season['mean_ci_upper']

# Model comparison
print("\nAIC Comparison:")
print(f"AIC with seasonality:    {logit_result.aic:.2f}")
print(f"AIC without seasonality: {logit_result_without_season.aic:.2f}")

# # Convert to odds ratios
# odds_ratios = {}
# for param, values in logit_result.items():
#     or_val = np.exp(values['coef'])
#     or_ci_lower = np.exp(values['ci_lower'])
#     or_ci_upper = np.exp(values['ci_upper'])
    
#     odds_ratios[param] = {
#         'OR': or_val,
#         'CI_lower': or_ci_lower,
#         'CI_upper': or_ci_upper,
#         'CI_formatted': f"{or_val:.2f} ({or_ci_lower:.2f}-{or_ci_upper:.2f})",
#         'p_value': values['p']
#     }
    
#     print(f"{param:20s}: OR = {or_val:.3f}, 95% CI: {or_ci_lower:.3f}-{or_ci_upper:.3f}, p = {values['p']:.3f}")

In [ ]:
# --- Aggregate and plot ---
summary_df = df.groupby('year').agg(
    actual_rate=('MME_above_50', 'mean'),
    predicted_rate=('predicted_prob', 'mean'),
    ci_lower=('ci_lower', 'mean'),
    ci_upper=('ci_upper', 'mean'),
    count=('MME_above_50', 'count'),
    predicted_rate_without_season=('predicted_prob_without_season', 'mean'),
    ci_lower_without_season=('ci_lower_without_season', 'mean'),
    ci_upper_without_season=('ci_upper_without_season', 'mean'),
).reset_index()

plt.figure(figsize=(12, 6))
plt.plot(summary_df['year'], summary_df['actual_rate'], label='Actual Opioid Rate', marker='o', color='green')
# Add intervention lines
plt.axvline(cutoff1, color='red', linestyle=':', label='2015 Intervention')
plt.axvline(cutoff2, color='red', linestyle=':', label='2019 Intervention')
# # With Seasons (fit line and confidence interval)
# plt.plot(summary_df['year'], summary_df['predicted_rate'], label='Predicted Rate (Seasons)', linestyle='--', color='blue')
# plt.fill_between(summary_df['year'], summary_df['ci_lower'], summary_df['ci_upper'], color='blue', alpha=0.2, label='95% CI (Seasons)')
# Without Seasons (fit line and confidence interval)
plt.plot(summary_df['year'], summary_df['predicted_rate_without_season'], label='Predicted (No Seasons)', linestyle='-.', color='orange')
plt.fill_between(summary_df['year'], summary_df['ci_lower_without_season'], summary_df['ci_upper_without_season'], color='orange', alpha=0.2, label='95% CI (No Seasons)')



plt.title('High-dosage (<=50 MME) Opioid Prescription Rate Over Time')
plt.ylabel('Rate')
plt.xlabel('Quarter')
plt.legend()
plt.grid(True)
plt.minorticks_on()
plt.tight_layout()
plt.savefig('high_dosage_annual_opioid_prescription_rate.tiff', dpi=300, bbox_inches='tight')
plt.show()


In [ ]:
# Get residuals from the fitted model
residuals = logit_result.resid_response

# --- Durbin-Watson statistic ---
dw_stat = durbin_watson(residuals)
print(f'Durbin-Watson statistic: {dw_stat:.2f}')

# --- ACF and PACF plots ---
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
plot_acf(residuals, lags=24, ax=ax[0])
plot_pacf(residuals, lags=24, ax=ax[1])
ax[0].set_title('ACF of Residuals')
ax[1].set_title('PACF of Residuals')
plt.tight_layout()
plt.show()


In [ ]:
# Coeff of Variation
print(np.std(summary_df['actual_rate'])/np.mean(summary_df['actual_rate']))
